# Test dei modelli FPViT a 224 px

Valuta i modelli in `trained_models_224/` sui test set di riferimento, **mai usati durante il training né per scegliere il checkpoint**:

| test set | immagini | cosa misura |
| --- | --- | --- |
| **DermaMNIST-C test** | 1227 | stessa distribuzione del training (HAM10000), lesioni mai viste |
| **DermaMNIST-E test** | 1511 | test ISIC 2018: **esterno**, lesioni e condizioni di acquisizione diverse |

Per ogni modello, e per l'**ensemble** dei seed (media delle probabilità), calcola accuracy, balanced accuracy, macro-F1 e macro-AUC, le metriche per classe e la **matrice di confusione**. Come riferimento valuta anche `baseline_paper` (FPViT a 28 px sullo split ufficiale) sugli stessi test: le lesioni di questi test non sono nel suo train, quindi il confronto è pulito.

**Da ricordare:** guardare il test e poi cambiare configurazione o scegliere il modello in base a questi numeri lo trasforma in una seconda validation. Le scelte si fanno sulla validation (cella 8 del notebook Colab); qui si misura soltanto.

Si esegue dalla cartella `notebooks/` con il `.venv` del progetto. Al primo avvio scarica da Zenodo i test set in `~/.medmnist`, con verifica MD5: circa 2,4 GB a 224 px e 45 MB a 28 px. Se li hai già scaricati altrove, imposta `DATA_ROOT`.

## 1 — Setup

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_recall_fscore_support, roc_auc_score)

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT))
from fpvit.dataset import dataset_file, eval_transform_for, load_split, model_input
from fpvit.engine import resolve_device
from fpvit.zoo import build_from_config

MODELS_DIR = PROJECT / "trained_models_224"
BASELINE   = PROJECT / "runs test solo training pyramid" / "baseline_paper" / "best_model.pt"
OUT_DIR    = MODELS_DIR / "test_results"
DATA_ROOT  = None                    # None = ~/.medmnist
TEST_SETS  = ["dermamnist_c", "dermamnist_e"]
INCLUDE_BASELINE = True
BATCH_SIZE = 64

device = resolve_device("auto")
OUT_DIR.mkdir(exist_ok=True)
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_NAMES = ["actinic keratoses / intraep. carcinoma", "basal cell carcinoma",
               "benign keratosis-like", "dermatofibroma", "melanoma",
               "melanocytic nevi", "vascular lesions"]
print("progetto:", PROJECT, "| device:", device)

## 2 — I modelli e cosa dice la validation

In [ ]:
def load_model(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = build_from_config(ckpt["config"]).to(device).eval()
    model.load_state_dict(ckpt["model_state_dict"])
    return model, ckpt

models, rows = {}, []
for path in sorted(MODELS_DIR.glob("*/best_model.pt")):
    name = path.parent.name
    model, ckpt = load_model(path)
    cfg, vm = ckpt["config"], ckpt["val_metrics"]
    rec = json.loads((path.parent / "experiment_record.json").read_text())
    models[name] = {"model": model, "cfg": cfg}
    rows.append({"modello": name, "dataset": cfg.get("dataset"), "input": f"{model_input(cfg)[0]} px",
                 "pretrained": cfg.get("pretrained"), "seed": cfg.get("seed"),
                 "stop": rec["stop_reason"], "epoche": rec["epochs_completed"], "epoca scelta": ckpt["epoch"],
                 "val bal_acc": vm["balanced_acc"], "val macro_auc": vm["macro_auc"], "val acc": vm["acc"]})
    assert cfg.get("dataset") == "dermamnist_c", f"{name}: addestrato su {cfg.get('dataset')}, non su C"

if INCLUDE_BASELINE and BASELINE.exists():
    model, ckpt = load_model(BASELINE)
    models["baseline_paper_28"] = {"model": model, "cfg": ckpt["config"]}
    rows.append({"modello": "baseline_paper_28", "dataset": "dermamnist (ufficiale)", "input": "28 px",
                 "pretrained": False, "seed": ckpt["config"].get("seed"), "epoca scelta": ckpt["epoch"],
                 "val bal_acc": ckpt["val_metrics"]["balanced_acc"],
                 "val macro_auc": ckpt["val_metrics"]["macro_auc"], "val acc": ckpt["val_metrics"]["acc"]})

display(pd.DataFrame(rows).round(3))
print("Nota: la validation di baseline_paper e' quella ufficiale (con leakage), non C-val: i numeri 'val' non sono confrontabili tra le righe.")

## 3 — Previsioni sui test set

Ogni modello riceve le immagini alla **sua** risoluzione e con la **sua** normalizzazione, entrambe lette dal checkpoint: 224 px con statistiche ImageNet per i modelli pre-addestrati, 28 px per `baseline_paper`. L'**ensemble** è la media delle probabilità dei modelli a 224 px, tutti con lo stesso peso.

In [ ]:
@torch.no_grad()
def predict_probs(model, cfg, dataset):
    img_size, _ = model_input(cfg)
    ds = load_split("test", eval_transform_for(cfg), dataset, img_size, DATA_ROOT)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    probs = [torch.softmax(model(x.to(device)), 1).cpu() for x, _ in loader]
    return torch.cat(probs).numpy(), ds.labels.reshape(-1).astype(int)

PROBS, LABELS = {}, {}
for ds_name in TEST_SETS:
    PROBS[ds_name] = {}
    for name, m in models.items():
        p, y = predict_probs(m["model"], m["cfg"], ds_name)
        PROBS[ds_name][name] = p
        if ds_name in LABELS:
            assert (LABELS[ds_name] == y).all(), "etichette diverse tra risoluzioni: stesso ordine atteso"
        LABELS[ds_name] = y
        print(f"{ds_name:13s} {name:28s} {len(y)} immagini")
    members = [n for n in PROBS[ds_name] if n != "baseline_paper_28"]
    if len(members) > 1:
        PROBS[ds_name]["ensemble_224"] = np.mean([PROBS[ds_name][n] for n in members], axis=0)

for ds_name, y in LABELS.items():
    print(f"\n{ds_name}: immagini per classe", dict(zip(CLASSES, np.bincount(y, minlength=7).tolist())))

## 4 — Metriche aggregate

In [ ]:
def metrics(y, p):
    pred = p.argmax(1)
    try:
        auc = roc_auc_score(y, p, multi_class="ovr", average="macro", labels=list(range(7)))
    except ValueError:
        auc = np.nan
    return {"accuracy": accuracy_score(y, pred), "balanced_acc": balanced_accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro", zero_division=0), "macro_auc": auc}

summary = pd.DataFrame([{"test": ds, "modello": name, **metrics(LABELS[ds], p)}
                        for ds in PROBS for name, p in PROBS[ds].items()])
display(summary.set_index(["test", "modello"]).round(3))
summary.to_csv(OUT_DIR / "summary.csv", index=False)

## 5 — Metriche per classe

Con classi così sbilanciate (i nevi sono circa due terzi) l'accuracy da sola inganna: la recall per classe dice se le classi rare vengono riconosciute davvero. Il *support* è il numero di immagini della classe nel test.

In [ ]:
per_class = []
for ds in PROBS:
    y = LABELS[ds]
    for name, p in PROBS[ds].items():
        prec, rec, f1, sup = precision_recall_fscore_support(y, p.argmax(1), labels=list(range(7)), zero_division=0)
        for c in range(7):
            onehot = (y == c).astype(int)
            auc_c = roc_auc_score(onehot, p[:, c]) if 0 < onehot.sum() < len(y) else np.nan
            per_class.append({"test": ds, "modello": name, "classe": CLASSES[c], "precision": prec[c],
                              "recall": rec[c], "f1": f1[c], "auc": auc_c, "support": int(sup[c])})
per_class = pd.DataFrame(per_class)
per_class.to_csv(OUT_DIR / "per_class.csv", index=False)

for ds in PROBS:
    print(f"\n=== {ds} — recall per classe ===")
    display(per_class[per_class.test == ds].pivot(index="modello", columns="classe", values="recall")[CLASSES].round(3))

In [ ]:
# Recall per classe: un gruppo di barre per classe, una barra per modello.
# Colori categoriali in ordine fisso; il colore segue il modello in tutti i grafici.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
names = list(PROBS[TEST_SETS[0]])
color = {n: SERIES[i % len(SERIES)] for i, n in enumerate(names)}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"

fig, axes = plt.subplots(len(PROBS), 1, figsize=(12, 3.6 * len(PROBS)), sharey=True)
axes = np.atleast_1d(axes)
width = 0.8 / len(names)
for ax, ds in zip(axes, PROBS):
    sub = per_class[per_class.test == ds]
    for i, n in enumerate(names):
        vals = sub[sub.modello == n].set_index("classe").loc[CLASSES, "recall"].values
        x = np.arange(7) + (i - (len(names) - 1) / 2) * width
        ax.bar(x, vals, width * 0.92, color=color[n], label=n, zorder=2)
    sup = sub.drop_duplicates("classe").set_index("classe").loc[CLASSES, "support"]
    ax.set_xticks(np.arange(7), [f"{c}\n(n={s})" for c, s in zip(CLASSES, sup)], color=INK2)
    ax.set_ylim(0, 1); ax.set_ylabel("recall", color=INK2); ax.set_title(ds, loc="left", color=INK)
    ax.grid(axis="y", color=GRID, zorder=0); ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors=INK2)
axes[0].legend(ncol=len(names), frameon=False, loc="lower left", bbox_to_anchor=(0, 1.08))
plt.tight_layout(); plt.savefig(OUT_DIR / "recall_per_class.png", dpi=150); plt.show()

## 6 — Matrici di confusione

Righe = classe vera, colonne = classe predetta. Il colore e la percentuale sono **normalizzati per riga**: la diagonale è la recall di ogni classe, e fuori diagonale si legge dove finiscono gli errori. Il numero tra parentesi è il conteggio di immagini. Una matrice per test set e per modello: `MATRIX_MODELS` sceglie quali mostrare.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
BLUES = LinearSegmentedColormap.from_list("seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
MATRIX_MODELS = [n for n in names if n != "baseline_paper_28"] + (["baseline_paper_28"] if "baseline_paper_28" in names else [])

def plot_confusion(ax, y, pred, title):
    cm = confusion_matrix(y, pred, labels=list(range(7)))
    norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(norm, cmap=BLUES, vmin=0, vmax=1)
    for i in range(7):
        for j in range(7):
            ink = "white" if norm[i, j] > 0.55 else INK
            ax.text(j, i, f"{norm[i, j]:.0%}\n({cm[i, j]})", ha="center", va="center", fontsize=7.5, color=ink)
    ax.set_xticks(range(7), CLASSES, color=INK2); ax.set_yticks(range(7), CLASSES, color=INK2)
    ax.set_xlabel("predetta", color=INK2); ax.set_ylabel("vera", color=INK2)
    ax.set_title(title, loc="left", fontsize=10, color=INK)
    for s in ax.spines.values(): s.set_visible(False)
    return cm

CONF = {}
for ds in PROBS:
    k = len(MATRIX_MODELS)
    fig, axes = plt.subplots(1, k, figsize=(5.2 * k, 5))
    for ax, n in zip(np.atleast_1d(axes), MATRIX_MODELS):
        CONF[(ds, n)] = plot_confusion(ax, LABELS[ds], PROBS[ds][n].argmax(1), f"{ds} · {n}")
    plt.tight_layout(); plt.savefig(OUT_DIR / f"confusion_{ds}.png", dpi=150); plt.show()

## 7 — Gli errori più importanti

Due confusioni contano più delle altre in dermatologia:
- un **melanoma predetto come nevo**: un tumore maligno scambiato per benigno;
- un **nevo predetto come melanoma**: un falso allarme, meno grave ma frequente.

Per ogni test set, conteggio delle due confusioni e le immagini sbagliate con la confidenza più alta, per il modello scelto in `SHOW_MODEL`.

In [ ]:
MEL, NV = CLASSES.index("mel"), CLASSES.index("nv")
rows = []
for (ds, n), cm in CONF.items():
    rows.append({"test": ds, "modello": n, "melanomi": int(cm[MEL].sum()),
                 "mel -> nv": int(cm[MEL, NV]), "mel -> nv %": cm[MEL, NV] / max(cm[MEL].sum(), 1),
                 "nevi": int(cm[NV].sum()), "nv -> mel": int(cm[NV, MEL]),
                 "nv -> mel %": cm[NV, MEL] / max(cm[NV].sum(), 1)})
display(pd.DataFrame(rows).round(3))

In [ ]:
SHOW_MODEL = "ensemble_224" if "ensemble_224" in names else names[0]
N_SHOW = 12

for ds in PROBS:
    img_size = 224
    raw = load_split("test", None, ds, img_size, DATA_ROOT)        # immagini originali, non normalizzate
    y, p = LABELS[ds], PROBS[ds][SHOW_MODEL]
    pred, conf = p.argmax(1), p.max(1)
    wrong = np.where(pred != y)[0]
    worst = wrong[np.argsort(-conf[wrong])][:N_SHOW]
    fig, axes = plt.subplots(2, N_SHOW // 2, figsize=(2.3 * N_SHOW // 2, 5.4))
    for ax, i in zip(axes.flat, worst):
        ax.imshow(raw.imgs[i]); ax.axis("off")
        ax.set_title(f"vera {CLASSES[y[i]]}\npred {CLASSES[pred[i]]} {conf[i]:.0%}", fontsize=8, color=INK)
    fig.suptitle(f"{ds} · {SHOW_MODEL}: errori con confidenza più alta ({len(wrong)} errori su {len(y)})",
                 x=0.01, ha="left", color=INK)
    plt.tight_layout(); plt.show()

## 8 — Salvataggio

In [ ]:
report = {
    "models": {n: {k: m["cfg"].get(k) for k in ("dataset", "img_size", "pretrained", "norm", "seed")}
               for n, m in models.items()},
    "test_sets": {ds: {"images": int(len(y)), "per_class": np.bincount(y, minlength=7).tolist()}
                  for ds, y in LABELS.items()},
    "metrics": summary.to_dict(orient="records"),
    "confusion_matrices": {f"{ds}|{n}": cm.tolist() for (ds, n), cm in CONF.items()},
    "class_order": CLASSES,
}
(OUT_DIR / "test_report.json").write_text(json.dumps(report, indent=2, default=float))
for ds in PROBS:
    df = pd.DataFrame({"index": np.arange(len(LABELS[ds])), "true": [CLASSES[c] for c in LABELS[ds]]})
    for n, p in PROBS[ds].items():
        df[f"pred_{n}"] = [CLASSES[c] for c in p.argmax(1)]
        df[f"conf_{n}"] = p.max(1).round(4)
    df.to_csv(OUT_DIR / f"predictions_{ds}.csv", index=False)
print("salvato in", OUT_DIR)
for f in sorted(OUT_DIR.iterdir()): print("  ", f.name)

## 9 — Confronto di tutti i modelli su un test set comune

Qui si confrontano **tutti i modelli disponibili nel progetto**, con **un punteggio per modello**:
- i modelli a 224 px di `trained_models_224/`;
- i modelli a 28 px che il testing agent usa oggi (`runs/`, `runs_agent/`, `runs test solo training pyramid/`, `models_promoted/`);
- due ensemble: quello a 224 px (media dei seed) e **quello attuale dell'agente**, cioè i modelli che il testing agent usa oggi (senza quelli in `ensemble_exclusions.json`), con il voto soft pesato per skill su DermaMNIST-C val, come in `predict_agent/voting.py`. Qui ogni modello riceve l'immagine alla sua risoluzione nativa, come per un'immagine caricata a 224 px: votano tutti.

I modelli esclusi dall'ensemble (`probe`, `smoke`) compaiono comunque nel confronto, come modelli singoli. Un modello promosso è una copia del run originale: compare una volta sola.

**Il test set comune.** Deve essere senza leakage per *ogni* modello, compresi quelli addestrati sullo split ufficiale. Verificato sui metadati di Abhishek et al.:
- **DermaMNIST-E test** (ISIC 2018, 1511 immagini): nessuna immagine in comune con HAM10000, da cui provengono tutti i train del progetto. È il test **principale**: esterno, il più grande, e indipendente sia dal training sia da ogni scelta fatta sulla validation.
- **DermaMNIST-C test** (1227 immagini): tutte immagini del test ufficiale, e nessuna delle sue 1149 lesioni compare nel train o nella validation ufficiali. È il test **secondario**, con la stessa distribuzione del training.

Ogni modello riceve le immagini alla sua risoluzione (28 o 224 px), ricavate dagli originali, con la sua normalizzazione.

**Robustezza.** Ogni punteggio ha un **intervallo di confidenza al 95% con bootstrap**: si ricampiona il test con reinserimento `N_BOOT` volte, gli stessi campioni per tutti i modelli. I confronti tra modelli sono quindi *appaiati*: la differenza con l'ensemble attuale ha un proprio intervallo, e se non contiene lo 0 la differenza non è dovuta al caso.

**Le metriche** (tutte su 7 classi):
- **accuracy**, la frazione di immagini classificate correttamente. Con i nevi a circa due terzi del test, da sola inganna;
- **macro-F1**, la media degli F1 per classe, dove ogni classe pesa uguale;
- **macro-AUC**, la media delle AUC one-vs-rest per classe. Misura quanto bene le probabilità ordinano i casi, indipendentemente dalla soglia;
- come colonna in più, la **balanced accuracy** (media delle recall), la metrica su cui il progetto seleziona i modelli.

In [ ]:
import hashlib
from predict_agent.models import DEFAULT_MODEL_ROOTS, ModelZoo

COMMON_TESTS = ["dermamnist_e", "dermamnist_c"]    # il primo e' il test principale
N_BOOT = 1000
REFERENCE = "ensemble_agente_attuale"              # rispetto a chi si calcolano le differenze

# L'ensemble che il testing agent usa oggi (radici ed esclusioni di default)...
agent_zoo = ModelZoo(device="cpu")
AGENT = set(agent_zoo.cards)
# ...e tutti i checkpoint disponibili, esclusi compresi, per il confronto dei singoli modelli.
zoo = ModelZoo(roots=list(DEFAULT_MODEL_ROOTS) + [MODELS_DIR], device="cpu", exclusions={})

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""): h.update(block)
    return h.hexdigest()

ALL, seen = {}, {}
for name, card in sorted(zoo.cards.items(), key=lambda kv: kv[0] not in AGENT):   # prima i nomi dell'agente
    digest = sha(card.path)
    if digest in seen:                                  # copia promossa dello stesso checkpoint
        print(f"{name}: stesso checkpoint di {seen[digest]}, contato una volta")
        continue
    seen[digest] = name
    model, _ = load_model(card.path)
    ALL[name] = {"model": model, "cfg": card.config, "skill": card.skill, "in_agent": name in AGENT,
                 "in_224_folder": Path(card.path).parent.parent == MODELS_DIR}
for s in zoo.skipped:
    print("non valutabile:", s["name"], "-", s["reason"])

display(pd.DataFrame([{"modello": n, "arch": m["cfg"].get("arch") or "fpvit",
                       "train": m["cfg"].get("dataset") or "dermamnist (ufficiale)",
                       "input": f"{model_input(m['cfg'])[0]} px", "nell'agente oggi": m["in_agent"],
                       "peso nel voto": round(m["skill"], 3) if m["in_agent"] else None}
                      for n, m in ALL.items()]))

In [ ]:
_split_cache = {}
@torch.no_grad()
def probs_on(name, dataset):
    m = ALL[name]
    img_size, norm = model_input(m["cfg"])
    key = (dataset, img_size, norm)
    if key not in _split_cache:                                   # ogni test caricato una volta per input
        _split_cache[key] = load_split("test", eval_transform_for(m["cfg"]), dataset, img_size, DATA_ROOT)
    ds = _split_cache[key]
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    p = torch.cat([torch.softmax(m["model"](x.to(device)), 1).cpu() for x, _ in loader]).numpy()
    return p, ds.labels.reshape(-1).astype(int)

CP, CY = {}, {}
for ds in COMMON_TESTS:
    CP[ds] = {}
    for name in ALL:
        CP[ds][name], y = probs_on(name, ds)
        assert ds not in CY or (CY[ds] == y).all(), "etichette diverse tra 28 e 224 px"
        CY[ds] = y
    in_224 = [n for n, m in ALL.items() if m["in_224_folder"] or
              (m["in_agent"] and model_input(m["cfg"])[0] == 224)]      # i seed a 224 px, ovunque stiano
    in_agent = [n for n, m in ALL.items() if m["in_agent"]]
    if len(in_224) > 1:
        CP[ds]["ensemble_224"] = np.mean([CP[ds][n] for n in in_224], axis=0)
    if in_agent:                                                   # stesso voto soft di predict_agent/voting.py
        w = np.array([ALL[n]["skill"] for n in in_agent])
        CP[ds][REFERENCE] = np.tensordot(w / w.sum(), np.stack([CP[ds][n] for n in in_agent]), axes=1)
    print(f"{ds}: {len(CY[ds])} immagini, {len(CP[ds])} modelli/ensemble")

In [ ]:
def scores(y, p):
    pred = p.argmax(1)
    try:
        auc = roc_auc_score(y, p, multi_class="ovr", average="macro", labels=list(range(7)))
    except ValueError:                        # un ricampionamento senza una classe
        auc = np.nan
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
            "macro_auc": auc, "balanced_acc": balanced_accuracy_score(y, pred)}

METRICS = ["accuracy", "macro_f1", "macro_auc", "balanced_acc"]
rng = np.random.default_rng(0)
BOOT, POINT = {}, {}
for ds in COMMON_TESTS:
    y = CY[ds]
    idx = [rng.integers(0, len(y), len(y)) for _ in range(N_BOOT)]    # stessi campioni per tutti i modelli
    for name, p in CP[ds].items():
        POINT[(ds, name)] = scores(y, p)
        BOOT[(ds, name)] = pd.DataFrame([scores(y[i], p[i]) for i in idx])

def ci(values):
    lo, hi = np.nanpercentile(values, [2.5, 97.5])
    return lo, hi

rows = []
for (ds, name), pt in POINT.items():
    row = {"test": ds, "modello": name}
    for k in METRICS:
        lo, hi = ci(BOOT[(ds, name)][k])
        row[k] = pt[k]; row[f"{k} IC95"] = f"[{lo:.3f}, {hi:.3f}]"
    rows.append(row)
SCORES = pd.DataFrame(rows)
SCORES.to_csv(OUT_DIR / "common_test_scores.csv", index=False)
for ds in COMMON_TESTS:
    print(f"\n=== {ds} ({len(CY[ds])} immagini){' - TEST PRINCIPALE' if ds == COMMON_TESTS[0] else ''} ===")
    t = SCORES[SCORES.test == ds].drop(columns="test").sort_values("macro_f1", ascending=False)
    display(t.set_index("modello").round(3))

In [ ]:
# Punteggio con intervallo al 95%: un pannello per metrica, stesso ordine dei modelli
# in tutti i pannelli (per macro-F1 sul test principale), una sola serie quindi un solo colore.
MAIN = COMMON_TESTS[0]
order = SCORES[SCORES.test == MAIN].sort_values("macro_f1").modello.tolist()
for ds in COMMON_TESTS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 0.45 * len(order) + 1.6), sharey=True)
    for ax, k in zip(axes, ["accuracy", "macro_f1", "macro_auc"]):
        for i, n in enumerate(order):
            pt = POINT[(ds, n)][k]; lo, hi = ci(BOOT[(ds, n)][k])
            ax.plot([lo, hi], [i, i], color="#86b6ef", lw=2, solid_capstyle="round", zorder=2)
            ax.plot(pt, i, "o", ms=8, color="#2a78d6", mec="#fcfcfb", mew=2, zorder=3)
            ax.text(hi + 0.01, i, f"{pt:.3f}", va="center", fontsize=8, color=INK2)
        ax.set_title(k, loc="left", color=INK); ax.set_xlim(0, 1.08)
        ax.grid(axis="x", color=GRID, zorder=0); ax.spines[["top", "right", "left"]].set_visible(False)
        ax.tick_params(colors=INK2, left=False)
    axes[0].set_yticks(range(len(order)), order, color=INK)
    fig.suptitle(f"{ds}: punteggio e intervallo di confidenza al 95% (bootstrap, {N_BOOT} campioni)",
                 x=0.01, ha="left", color=INK)
    plt.tight_layout(); plt.savefig(OUT_DIR / f"common_test_{ds}.png", dpi=150); plt.show()

### Differenza rispetto all'ensemble attuale dell'agente

Per ogni modello: punteggio del modello meno punteggio di `ensemble_agente_attuale`, calcolato **sugli stessi campioni bootstrap**. Se l'intervallo al 95% non contiene lo 0, il modello è significativamente migliore (differenza positiva) o peggiore (negativa) di quello che l'agente usa oggi.

In [ ]:
rows = []
for ds in COMMON_TESTS:
    if (ds, REFERENCE) not in BOOT: continue
    ref = BOOT[(ds, REFERENCE)]
    for name in CP[ds]:
        if name == REFERENCE: continue
        row = {"test": ds, "modello": name}
        for k in ["accuracy", "macro_f1", "macro_auc"]:
            d = BOOT[(ds, name)][k] - ref[k]
            lo, hi = ci(d)
            row[f"Δ {k}"] = POINT[(ds, name)][k] - POINT[(ds, REFERENCE)][k]
            row[f"Δ {k} IC95"] = f"[{lo:+.3f}, {hi:+.3f}]" + (" *" if lo > 0 or hi < 0 else "")
        rows.append(row)
DELTA = pd.DataFrame(rows)
DELTA.to_csv(OUT_DIR / "common_test_delta_vs_agent.csv", index=False)
for ds in COMMON_TESTS:
    print(f"\n=== {ds}: differenza da {REFERENCE} (* = intervallo che esclude lo 0) ===")
    display(DELTA[DELTA.test == ds].drop(columns="test").set_index("modello")
            .sort_values("Δ macro_f1", ascending=False).round(3))